# 🛠️ Modul 03 Hands-on: Generalization, Tuning & Debugging

Di modul ini, Anda akan mempraktikkan keterampilan bertahan hidup insinyur AI:
1. **The Overfitting Sandbox**: Menciptakan kondisi overfit parah dan mengamati kurva train vs val loss.
2. **The Antidotes in Action**: Menerapkan **Dropout**, **Weight Decay (AdamW)**, dan **Batch Normalization** untuk menyembuhkan overfitting.
3. **The Symmetry Sabotage**: Membuktikan secara empiris kegagalan total inisialisasi bobot nol.
4. **Break & Fix Challenge**: Menyelesaikan skenario debugging nyata di mana model gagal belajar.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))
from shared.visualizer import plot_decision_boundary

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Device aktif: {device}")

## 1. The Overfitting Sandbox: Menghafal Data Latihan
Kita sengaja membuat dataset latihan yang sangat kecil (hanya 60 sampel) dengan noise tinggi, lalu melatih model raksasa (256 neuron) tanpa regularisasi sama sekali.

In [ ]:
# Buat data kecil ber-noise tinggi
X_raw, y_raw = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(X_raw, y_raw, train_size=60, random_state=42)

X_train = torch.tensor(X_train_np, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1).to(device)
X_val = torch.tensor(X_val_np, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val_np, dtype=torch.float32).unsqueeze(1).to(device)

# Model raksasa yang rentan overfit (Over-parameterized)
class OverfitModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

model_overfit = OverfitModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_overfit.parameters(), lr=0.01)

train_losses, val_losses = [], []
for epoch in range(1, 401):
    model_overfit.train()
    optimizer.zero_grad()
    out_train = model_overfit(X_train)
    loss_train = criterion(out_train, y_train)
    loss_train.backward()
    optimizer.step()
    
    model_overfit.eval()
    with torch.no_grad():
        out_val = model_overfit(X_val)
        loss_val = criterion(out_val, y_val)
        
    train_losses.append(loss_train.item())
    val_losses.append(loss_val.item())

# Plot kurva Overfitting
plt.figure(figsize=(9, 5))
plt.plot(train_losses, label="Train Loss (Terus Turun Mendekati Nol)", color="blue", linewidth=2)
plt.plot(val_losses, label="Validation Loss (Mulai Naik Tajam / Overfit!)", color="red", linewidth=2)
plt.title("Anatomi Overfitting: Gap Lebar Antara Train dan Validation", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.ylim(0, 1.5)
plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# Visualisasikan decision boundary yang bergelombang tajam membungkus noise
plot_decision_boundary(model_overfit, X_train, y_train, title="Decision Boundary Overfit (Membungkus Noise)")

## 2. The Antidotes: Menjinakkan Overfitting dengan Dropout & Weight Decay
Sekarang kita bangun model yang sama, tetapi kita tambahkan:
1. **`nn.Dropout(0.3)`** setelah setiap layer aktivasi.
2. **`weight_decay=1e-3`** pada optimizer AdamW.

In [ ]:
class RegularizedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),  # Matikan 30% neuron acak saat training
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

model_reg = RegularizedModel().to(device)
optimizer_reg = optim.AdamW(model_reg.parameters(), lr=0.01, weight_decay=1e-3)

train_losses_reg, val_losses_reg = [], []
for epoch in range(1, 401):
    model_reg.train()  # AKTIFKAN Dropout & BatchNorm tracking
    optimizer_reg.zero_grad()
    out_train = model_reg(X_train)
    loss_train = criterion(out_train, y_train)
    loss_train.backward()
    optimizer_reg.step()
    
    model_reg.eval()   # NONAKTIFKAN Dropout & Bekukan BatchNorm
    with torch.no_grad():
        out_val = model_reg(X_val)
        loss_val = criterion(out_val, y_val)
        
    train_losses_reg.append(loss_train.item())
    val_losses_reg.append(loss_val.item())

# Bandingkan kurva validasi
plt.figure(figsize=(9, 5))
plt.plot(val_losses, label="Val Loss Tanpa Regularisasi (Meledak)", color="red", linestyle="--", linewidth=1.8)
plt.plot(val_losses_reg, label="Val Loss Dengan Dropout + L2 (Stabil & Rendah)", color="green", linewidth=2.5)
plt.title("Penyembuhan Overfitting: Val Loss Menjadi Jauh Lebih Sehat!", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# Visualisasikan decision boundary yang kembali mulus dan rapi
plot_decision_boundary(model_reg, X_train, y_train, title="Decision Boundary Regularized (Mulus & General)")

## 3. The Symmetry Sabotage: Bahaya Inisialisasi Bobot Nol
Mari kita buktikan secara empiris apa yang terjadi jika semua bobot di-set ke nol ($W = 0$).

In [ ]:
model_zero = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 1)).to(device)

# PAKSA SEMUA BOBOT MENJADI NOL
with torch.no_grad():
    for param in model_zero.parameters():
        param.fill_(0.0)

print("Bobot awal layer pertama:\n", model_zero[0].weight[:3, :])

# Lakukan 1 kali forward dan backward pass
model_zero.train()
out = model_zero(X_train)
loss = criterion(out, y_train)
loss.backward()

print("\nGradien bobot layer pertama setelah backward:\n", model_zero[0].weight.grad[:3, :])
print("\n💡 Perhatikan: Semua baris gradien BERNILAI IDENTIK SAMA PERSIS!")
print("Ini membuktikan Symmetry Trap: seluruh neuron terjebak memperbarui bobot dengan nilai yang sama persis!")

## 4. 🏆 The Break & Fix Challenge

Di bawah ini ada kode training yang **RUSAK PARAH** karena salah satu bug paling klasik di PyTorch: programmer lupa memanggil `optimizer.zero_grad()`. Akibatnya, gradien terakumulasi dari epoch sebelumnya dan training menjadi kacau balau.

**Tugas Anda**:
1. Jalankan cell untuk melihat kegagalannya.
2. Perbaiki bug tersebut di bagian yang ditandai `# FIX DI SINI`.
3. Pastikan model berhasil mencapai loss $< 0.35$!

In [ ]:
model_challenge = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
opt = optim.Adam(model_challenge.parameters(), lr=0.02)

losses = []
for epoch in range(1, 101):
    model_challenge.train()
    
    # ========================================================
    # FIX DI SINI: Baris apa yang wajib dipanggil sebelum forward/backward?
    opt.zero_grad()  # <-- Kunci perbaikan!
    # ========================================================
    
    pred = model_challenge(X_train)
    loss = criterion(pred, y_train)
    loss.backward()
    opt.step()
    losses.append(loss.item())

print(f"Final Loss: {losses[-1]:.4f}")
if losses[-1] < 0.35:
    print("🎉 HEBAT! Model sembuh dan konvergen sempurna!")
else:
    print("❌ Masih ada bug, periksa kembali kode Anda!")